In [1]:
import sys
sys.path.insert(0, "/home/student/lisa_ma")

import json
import glob
import torch
import numpy as np
import pandas as pd
from pathlib import Path
from torch.utils.data import DataLoader
from sklearn.metrics import f1_score, matthews_corrcoef
from monai.transforms import Compose, NormalizeIntensity

from dataloaders.dataloader_VerSe import VerSeDataLoader, VerSeDataset, custom_collate
import datasets.VerSe.get_data_VerSe as get_data_VerSe
from models.model_densenet import DenseNetModel

DATASET_DIR  = "/home/student/lisa_ma/prepared"
EXCEL_PATH   = "/home/student/lisa_ma/datasets/VerSe/data_filter_joined.xlsx"
SPEC_DATA    = {"mean": [-111.3885], "std": [406.3665]}
DEVICE       = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")

2026-09-28 21:09:47.327113: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1790622587.342877 3710977 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1790622587.347772 3710977 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1790622587.361191 3710977 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1790622587.361203 3710977 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1790622587.361205 3710977 computation_placer.cc:177] computation placer alr

Using device: cuda


In [2]:
class _Normalize:
    """Val-only transform: normalization, no augmentation."""
    def __init__(self):
        self.t = Compose([NormalizeIntensity(
            subtrahend=SPEC_DATA["mean"], divisor=SPEC_DATA["std"], channel_wise=True
        )])
    def __call__(self, x):
        return self.t(x)

_transform = _Normalize()

# Compute pid_corrections once from the lower-bound spec (same for all classifiers).
LOWER_BOUND_SPEC = "/home/student/lisa_ma/results/classifier/test_glare_20072026_0111/spec.json"

with open(LOWER_BOUND_SPEC) as f:
    _lb_spec = json.load(f)
_lb_spec["dataset_dir"] = DATASET_DIR
_lb_spec["batch_size"] = 1

_dm = VerSeDataLoader(spec=_lb_spec, train_transforms=_transform, val_transforms=_transform, num_workers=0)
_dm.setup()
PID_CORRECTIONS = _dm.train_dataset.pid_corrections
print(f"pid_corrections loaded: {len(PID_CORRECTIONS)} PIDs")

# Load test files (shared across all classifiers)
TEST_FILES, _ = get_data_VerSe.get_filtered_files_across_dsnames(
    root_dir=DATASET_DIR,
    excel_path=EXCEL_PATH,
    split="test",
    glob_pattern="*.npz",
    pid_col="pid",
    dsname_col="dsname",
    verts_col="vert_label",
    check_complete=True,
    apply_excel_filter=True,
    extra_sacral_verts=frozenset(),
)
print(f"Test samples: {len(TEST_FILES)}")

Collected 19266 training files.
All expected vertebra files present.
PID corrections (sample):
  ctfu00610_ses-20091111_ce-pv: {8: 8, 9: 9, 10: 10, 11: 11, 12: 12, 13: 13, 14: 14, 15: 15, 16: 16, 17: 17, 18: 18, 19: 19, 20: 21, 21: 22, 22: 23, 23: 24, 28: 20}
  ctfu00897_ses-20170303_ce-art: {8: 8, 9: 9, 10: 10, 11: 11, 12: 12, 13: 13, 14: 14, 15: 15, 16: 16, 17: 17, 18: 18, 19: 19, 20: 21, 21: 22, 22: 23, 28: 20}
  ctfu01006_ses-20160915_ce-no: {8: 8, 9: 9, 10: 10, 11: 11, 12: 12, 13: 13, 14: 14, 15: 15, 16: 16, 17: 17, 18: 18, 19: 19, 20: 21, 21: 22, 22: 23, 23: 24, 24: 25, 28: 20}
  ID0049_ses-14082007_ce-pv: {8: 8, 9: 9, 10: 10, 11: 11, 12: 12, 13: 13, 14: 14, 15: 15, 16: 16, 17: 17, 18: 18, 20: 19, 21: 20, 22: 21, 23: 22, 24: 23}
  ID0067_ses-04062009_ce-pv: {7: 7, 8: 8, 9: 9, 10: 10, 11: 11, 12: 12, 13: 13, 14: 14, 15: 15, 16: 16, 17: 17, 18: 18, 20: 19, 21: 20, 22: 21, 23: 22, 24: 23, 25: 24}
  ID0297_ses-29102010_ce-pv: {8: 8, 9: 9, 10: 10, 11: 11, 12: 12, 13: 13, 14: 14, 15: 1

In [3]:
def evaluate_on_test_set(spec_path, epoch=7):
    spec_path = Path(spec_path)
    with open(spec_path) as f:
        spec = json.load(f)

    test_dataset = VerSeDataset(TEST_FILES, transform=_transform)
    test_dataset.pid_corrections = PID_CORRECTIONS
    test_loader = DataLoader(
        test_dataset, batch_size=8, shuffle=False, num_workers=4, collate_fn=custom_collate
    )

    weights_dir = spec_path.parent / "weights"
    checkpoint   = weights_dir / f"epoch_{epoch}"
    model = DenseNetModel(num_classes=4, spec=spec, weights_dir=str(weights_dir))
    model.load_state_dict(torch.load(checkpoint, map_location="cpu"))
    model.eval().to(DEVICE)

    all_preds, all_labels = [], []
    with torch.no_grad():
        for X, labels, _ in test_loader:
            preds = torch.argmax(model(X.to(DEVICE)), dim=1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    return {
        "weighted_f1": round(f1_score(all_labels, all_preds, average="weighted"), 4),
        "mcc":         round(matthews_corrcoef(all_labels, all_preds), 4),
        "n_test":      len(all_labels),
    }

In [4]:
def _find_spec(glob_pattern):
    """Return the spec.json path for the most recently created matching classifier dir."""
    matches = sorted(glob.glob(glob_pattern))
    if not matches:
        return None
    return matches[-1]  # latest timestamp suffix

CLASSIFIERS = [
    ("Lower bound",       "/home/student/lisa_ma/results/classifier/test_glare_20072026_0111/spec.json"),
    ("GLARE 0.2% correct", _find_spec("/home/student/lisa_ma/results/VerSe_classifier/glare_correction_frac02_*/spec.json")),
    ("GLARE 0.4% correct", _find_spec("/home/student/lisa_ma/results/VerSe_classifier/glare_correction_frac04_*/spec.json")),
    ("Upper bound",        _find_spec("/home/student/lisa_ma/results/VerSe_classifier/upper_bound_8ep_*/spec.json")),
]

for name, path in CLASSIFIERS:
    print(f"{name:22s}: {path}")

Lower bound           : /home/student/lisa_ma/results/classifier/test_glare_20072026_0111/spec.json
GLARE 0.2% correct    : /home/student/lisa_ma/results/VerSe_classifier/glare_correction_frac02_25092026_1959/spec.json
GLARE 0.4% correct    : /home/student/lisa_ma/results/VerSe_classifier/glare_correction_frac04_26092026_0122/spec.json
Upper bound           : /home/student/lisa_ma/results/VerSe_classifier/upper_bound_8ep_25092026_1348/spec.json


In [ ]:
rows = []
for name, spec_path in CLASSIFIERS:
    if spec_path is None or not Path(spec_path).exists():
        print(f"  SKIP {name}: spec not found")
        rows.append({"Classifier": name, "Weighted F1": None, "MCC": None})
        continue
    print(f"  Evaluating {name} ...", end=" ", flush=True)
    result = evaluate_on_test_set(spec_path, epoch=7)
    rows.append({"Classifier": name, "Weighted F1": result["weighted_f1"], "MCC": result["mcc"]})
    print(f"F1={result['weighted_f1']:.4f}  MCC={result['mcc']:.4f}  (n={result['n_test']})")

results_df = pd.DataFrame(rows)
print()
print(results_df.to_string(index=False))

  Evaluating Lower bound ... F1=0.9959  MCC=0.9926  (n=2211)
  Evaluating GLARE 0.2% correct ... F1=0.9933  MCC=0.9878  (n=2211)
  Evaluating GLARE 0.4% correct ... 